# Chicago Crime Data - Cleaning

Standard machine learning data cleaning steps, plus feature engineering (year, month, day of week, hour, etc.) for downstream modeling and EDA.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/raw/crimes.csv')
print("Initial shape:", df.shape)
df.head(3)

## 1. Drop Redundant Columns

Based on data understanding: Case Number, Location, X Coordinate, Y Coordinate, Updated On.

In [ ]:
cols_to_drop = ['Case Number', 'Location', 'X Coordinate', 'Y Coordinate', 'Updated On']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print("After dropping redundant columns:", df.shape)

## 2. Handle Missing Values

- Latitude/Longitude: Drop rows with missing coordinates (needed for spatial analysis)
- Location Description: Fill with "UNKNOWN" (categorical)
- Community Area: Fill with -1 for unknown

In [ ]:
print("Missing before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Drop rows with missing Lat/Lon (needed for mapping and location-based predictions)
n_before = len(df)
df = df.dropna(subset=['Latitude', 'Longitude'])
print(f"\nDropped {n_before - len(df)} rows with missing coordinates")

# Fill Location Description with UNKNOWN
df['Location Description'] = df['Location Description'].fillna('UNKNOWN')

# Fill Community Area with -1 (unknown)
df['Community Area'] = df['Community Area'].fillna(-1).astype(int)
print("\nMissing after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Parse Date and Add Temporal Features

Extract year, month, day of week, hour, quarter, weekend flag for timing analysis and modeling.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df = df.dropna(subset=['Date'])  # drop any unparseable dates

# Add temporal features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['DayOfWeekName'] = df['Date'].dt.day_name()
df['Hour'] = df['Date'].dt.hour
df['Quarter'] = df['Date'].dt.quarter
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6])

print("Sample with new columns:")
df[['Date', 'Year', 'Month', 'DayOfWeekName', 'Hour', 'Quarter', 'IsWeekend']].head()

## 4. Handle Duplicates

Remove duplicate records by ID.

In [ ]:
dup_count = df.duplicated(subset=['ID']).sum()
df = df.drop_duplicates(subset=['ID'])
print(f"Removed {dup_count} duplicate rows by ID. Shape: {df.shape}")

## 5. Ensure Correct Data Types

Convert Arrest/Domestic to int for ML; ensure Ward and Community Area are int.

In [ ]:
df['Arrest'] = df['Arrest'].astype(int)
df['Domestic'] = df['Domestic'].astype(int)
df['Ward'] = df['Ward'].fillna(-1).astype(int)
print("Data types check:")
df.dtypes

## 6. Handle Outliers (Coordinates)

Filter out points outside Chicago bounds. Chicago roughly: Lat 41.6-42.0, Lon -87.95 to -87.5.

In [ ]:
CHICAGO_LAT = (41.6, 42.1)
CHICAGO_LON = (-87.95, -87.5)
mask = (df['Latitude'].between(*CHICAGO_LAT)) & (df['Longitude'].between(*CHICAGO_LON))
outliers = (~mask).sum()
df = df[mask]
print(f"Removed {outliers} rows with coordinates outside Chicago. Shape: {df.shape}")

## 7. Save Cleaned Data

Store the cleaned dataset for EDA and modeling.

In [ ]:
Path('../data/processed').mkdir(parents=True, exist_ok=True)
df.to_csv('../data/processed/crimes_cleaned.csv', index=False)
print("Saved to ../data/processed/crimes_cleaned.csv")
print("Final shape:", df.shape)
df.head()